# Análise de dados na prática: investigando o Titanic

Neste notebook você vai **investigar** uma tabela de dados como um detetive: partindo de perguntas, não de ferramentas.

**O ciclo que vamos repetir o tempo todo:**

`pergunta → dados → limpeza → exploração → resposta → comunicação`

## Como usar este notebook (Google Colab)
1. Faça sua cópia: **Arquivo → Salvar uma cópia no Drive**. Sem isso, o que você fizer se perde ao fechar a aba.
2. Cada bloco cinza é uma **célula de código**. Clique nela e aperte **Shift + Enter** para rodar. O resultado aparece logo abaixo.
3. **Rode as células em ordem, de cima para baixo.** Se der erro dizendo que algo "não está definido", provavelmente você pulou uma célula. Use **Ambiente de execução → Executar tudo** para recomeçar.
4. Os blocos de texto (como este) são só para ler.

## Regra de ouro
Antes de rodar cada célula, **escreva sua hipótese** (pode ser no papel). O que você espera ver? Depois compare. Errar a hipótese é onde se aprende.

## Mini-glossário
| Termo | Significa |
|---|---|
| **DataFrame** (`df`) | A tabela. Cada linha é um registro, cada coluna é uma característica. |
| **Nulo / NaN** | Valor que não foi registrado (célula vazia). |
| **Média (`mean`)** de uma coluna 0/1 | Proporção de 1s. Se `survived` é 1 para quem sobreviveu, a média é a taxa de sobrevivência. |
| **Outlier** | Valor muito afastado dos demais. Pode ser erro ou pode ser real. |
| **Baseline** | O "chute óbvio" que qualquer modelo precisa superar. |

## Carregando os dados
O Titanic é um dataset clássico que já vem com a biblioteca `seaborn`. Cada linha é **um passageiro**.

In [ ]:
# Importamos as bibliotecas que vamos usar:
import pandas as pd              # manipular tabelas (DataFrames)
import seaborn as sns            # gráficos bonitos e alguns datasets de exemplo
import matplotlib.pyplot as plt  # base dos gráficos (usamos para mostrar/titular)

# Baixa o dataset do Titanic e guarda na variável df (de "DataFrame").
# Precisa de internet; no Colab isso já funciona.
df = sns.load_dataset("titanic")

# head() mostra as 5 primeiras linhas, só para dar uma olhada.
df.head()

**Leia a tabela antes de analisar.** Algumas colunas importantes:
- `survived`: 1 = sobreviveu, 0 = morreu (é o que queremos entender)
- `pclass`: classe do bilhete (1 = mais cara, 3 = mais barata)
- `sex`, `age`: sexo e idade
- `fare`: valor pago pela passagem
- `sibsp` / `parch`: irmãos/cônjuges e pais/filhos a bordo
- `embarked`, `embark_town`: porto de embarque
- `deck`: convés da cabine
- Algumas colunas (`class`, `who`, `alive`, `alone`…) são versões "traduzidas" ou derivadas de outras.

## Parte 0: conhecendo os dados
**Pergunta:** o que tem nessa tabela? Cada linha é o quê? Quantas linhas e colunas?

*Sua hipótese: quantos passageiros você acha que há?*

In [ ]:
# shape devolve (número de linhas, número de colunas)
print(df.shape)

# info() lista cada coluna, o tipo de dado e QUANTOS valores NÃO são nulos.
# Dica: se uma coluna tem menos valores que o total de linhas, ela tem nulos.
df.info()

**O que olhar:** compare a coluna `Non-Null Count` com o total de linhas. Quais colunas têm menos? Guarde essa lista, vamos voltar a ela na Parte 2.

In [ ]:
# describe() resume as colunas numéricas (média, mínimo, máximo...).
# include="all" inclui também as colunas de texto; .T "vira" a tabela de lado para ler melhor.
df.describe(include="all").T

**O que olhar:** em `age`, veja o `min` e o `max`: fazem sentido? Em `fare`, a média e o máximo estão muito distantes? Isso é um sinal de que a tarifa tem valores extremos (voltamos nisso na Parte 5).

## Parte 1: quantos?
**Pergunta:** quantos passageiros temos e qual a proporção de sobreviventes?

*Sua hipótese: mais da metade sobreviveu ou menos da metade?*

In [ ]:
# len(df) conta o número de linhas.
print("Passageiros:", len(df))

# Como survived vale 0 ou 1, a MÉDIA dela é exatamente a proporção de 1s,
# ou seja, a taxa de sobrevivência. Multiplicamos por 100 para virar porcentagem.
print("Taxa de sobrevivência:", round(df["survived"].mean() * 100, 1), "%")

**O que olhar:** a taxa geral é o nosso ponto de referência. Nas próximas partes, vamos perguntar se algum grupo está acima ou abaixo dela.

## Parte 2: o que está faltando?
**Pergunta:** quais colunas têm valores nulos? Isso é um problema?

*Sua hipótese: quais colunas você acha que têm mais nulos?*

In [ ]:
# isna() marca cada célula como nula (True) ou não (False).
# mean() de True/False dá a PROPORÇÃO de nulos por coluna; * 100 vira porcentagem.
nulos = df.isna().mean().sort_values(ascending=False) * 100

# Mostramos só as colunas que têm algum nulo (nulos > 0).
nulos[nulos > 0].round(1)

**Como ler:** o número é a porcentagem de linhas sem valor. Em geral, `deck` perde de longe (a maioria não tem convés registrado) e `age` fica em torno de 20%.

**Decisão de analista:** o que fazer com nulos? Há várias opções, cada uma com um custo:

| Opção | Vantagem | Risco |
|---|---|---|
| Remover as linhas com nulo | Simples | Perde dados e pode enviesar a amostra |
| Preencher (mediana, média) | Mantém todas as linhas | Inventa informação |
| Ignorar a coluna | Evita o problema | Perde um fator que pode ser importante |
| Ignorar o nulo só naquela análise | Não inventa nada | A amostra fica menor |

Não existe resposta certa universal. O que importa é **escolher conscientemente e registrar o porquê**.

In [ ]:
# Para a idade, vamos só IGNORAR os nulos nas análises que usarem essa coluna,
# sem inventar valores, e lembrar que a amostra ali é menor.
# describe() já ignora nulos: repare no "count" (menor que o total de passageiros).
df["age"].describe()

## Parte 3: quem sobreviveu?
**Pergunta:** sexo e classe influenciaram a chance de sobreviver?

*Sua hipótese: quem teve mais chance, homens ou mulheres? E qual classe?*

In [ ]:
# groupby("sex") separa os passageiros em grupos por sexo.
# ["survived"].mean() calcula, dentro de cada grupo, a taxa de sobrevivência.
print(df.groupby("sex")["survived"].mean().round(2))
print()

# Mesma ideia, agora separando por classe do bilhete.
print(df.groupby("pclass")["survived"].mean().round(2))

**O que olhar:** compare cada valor com a taxa geral da Parte 1. A diferença entre os grupos é grande? Grande o bastante para parecer coincidência?

In [ ]:
# barplot desenha a taxa média de sobrevivência (y) para cada classe (x).
# hue="sex" separa as barras por sexo, em cores diferentes.
sns.barplot(data=df, x="pclass", y="survived", hue="sex")

plt.title("Taxa de sobrevivência por classe e sexo")
plt.ylabel("Taxa de sobrevivência")
plt.show()

**Como ler o gráfico:** a altura da barra é a taxa de sobrevivência. As linhas pretas finas são a **incerteza** da estimativa (intervalo de confiança): quanto maior a linha, menos confiável a barra.

Pergunta para discutir: o padrão de sexo se repete nas três classes, ou muda?

## Parte 4: e a idade?
**Pergunta:** crianças tiveram mais chance de sobreviver? (Lembre dos nulos em `age`.)

*Sua hipótese: a regra "mulheres e crianças primeiro" aparece nos dados?*

In [ ]:
# pd.cut transforma uma idade contínua em FAIXAS.
# bins define os limites: (0,12], (12,18], (18,40], (40,60], (60,100].
# Quem tem age nulo continua nulo e fica de fora da faixa.
df["faixa_etaria"] = pd.cut(
    df["age"],
    bins=[0, 12, 18, 40, 60, 100],
    labels=["0-12", "13-18", "19-40", "41-60", "60+"],
)

# Para cada faixa: a taxa de sobrevivência (mean) e quantos passageiros (count).
# observed=True evita avisos do pandas sobre faixas vazias.
print(df.groupby("faixa_etaria", observed=True)["survived"].agg(["mean", "count"]).round(2))

Repare na coluna **`count`**: faixas com poucos passageiros dão taxas instáveis. Uma taxa de 70% baseada em 40 pessoas é muito menos confiável que 70% baseada em 400.

**Pergunta para discutir:** os limites das faixas (12, 18, 40...) foram escolha nossa. Mudar esses limites mudaria a conclusão? Teste!

## Parte 5: outliers
**Pergunta:** a tarifa (`fare`) tem valores estranhos?

*Sua hipótese: a tarifa mais alta é de qual classe?*

In [ ]:
# O boxplot mostra, para cada classe, a mediana (linha no meio da caixa),
# onde está a maioria dos valores (a caixa) e os pontos isolados (outliers).
sns.boxplot(data=df, x="pclass", y="fare")
plt.show()

# Ordena do maior para o menor valor de fare e mostra as 5 maiores tarifas.
df.sort_values("fare", ascending=False)[["pclass", "sex", "age", "fare"]].head()

Tarifas muito altas na 1ª classe são **reais**, não erro: havia suítes caríssimas. Já tarifa 0 pode ser dado faltante ou cortesia. **Outlier não é sinônimo de "apagar".** Primeiro entenda de onde ele veio.

In [ ]:
# Quantos passageiros têm tarifa igual a zero?
# (df["fare"] == 0) vira True/False por linha; sum() conta os True.
(df["fare"] == 0).sum()

**O que fazer com eles?** Anote uma decisão. Exemplo: "são poucos, não afetam a conclusão; mantive". O importante é saber que existem.

## Parte 6: correlação não é causa
**Pergunta:** a 1ª classe sobreviveu mais *porque* era 1ª classe?

Primeiro, veja: a composição por sexo é igual em todas as classes? Se não for, a comparação entre classes mistura dois efeitos.

In [ ]:
# crosstab conta quantos passageiros há em cada combinação classe x sexo.
# normalize="index" transforma em PROPORÇÃO dentro de cada linha (cada classe soma 1).
print(pd.crosstab(df["pclass"], df["sex"], normalize="index").round(2))
print()

# pivot_table cruza as duas variáveis e calcula a taxa de sobrevivência
# em cada combinação (classe x sexo) de uma vez.
print(df.pivot_table(index="pclass", columns="sex", values="survived", aggfunc="mean").round(2))

**Como ler:**
- A primeira tabela diz **quem estava em cada classe**. Se a 3ª classe tem proporcionalmente mais homens, parte da "desvantagem da 3ª classe" pode ser, na verdade, efeito do sexo.
- A segunda tabela compara **dentro de cada grupo**, o que separa os efeitos.

Comparar dentro de cada grupo ajuda, mas **ainda não prova causa**. Fatores como localização da cabine e acesso aos botes não estão nos dados.

> **Em uma frase:** dois fatores andarem juntos não significa que um causa o outro. Pode haver um terceiro fator escondido.

## Parte 7: uma pergunta que os dados não respondem
**Pergunta:** "Se o navio tivesse mais botes, quantos teriam sobrevivido?"

Discuta com o colega: o que faltaria para responder? Dados sobre botes, horários, posição das cabines...

Saber dizer **"esses dados não respondem isso"** também é resultado de análise, e é uma habilidade rara e valiosa.

## Parte 8: e se quisermos prever?
**Pergunta:** com poucas colunas, conseguimos prever quem sobreviveria?

Agora saímos da análise (entender o passado) para a previsão (palpitar sobre casos novos). Essa é a fronteira entre analista e cientista de dados.

*Sua hipótese: quantos % de acerto um modelo simples teria?*

In [ ]:
from sklearn.model_selection import train_test_split   # separa treino e teste
from sklearn.tree import DecisionTreeClassifier        # um modelo simples: árvore de decisão
from sklearn.metrics import accuracy_score             # % de acertos

# 1) Escolhemos as colunas (features) que o modelo vai usar para prever.
X = df[["pclass", "sex", "age", "fare"]].copy()

# 2) Modelos só entendem números: transformamos sexo em 0/1.
X["sex"] = (X["sex"] == "female").astype(int)

# 3) ATENÇÃO, DECISÃO CONSCIENTE: o modelo não aceita nulos, então AQUI preenchemos
#    a idade com a mediana. Na Parte 2 preferimos não inventar valores, porque ali
#    era uma análise descritiva. Para treinar um modelo precisamos de tabela completa.
#    Cada contexto pede uma decisão diferente, por isso ela deve ser registrada.
X["age"] = X["age"].fillna(X["age"].median())

# 4) y é o que queremos prever: sobreviveu ou não.
y = df["survived"]

# 5) Separamos 70% dos passageiros para TREINAR e 30% para TESTAR.
#    O modelo só é avaliado em quem ele nunca viu, senão seria "decorar a prova".
#    random_state fixa o sorteio para o resultado ser reproduzível.
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42)

# 6) Baseline: o chute óbvio é dizer que NINGUÉM sobreviveu (a maioria morreu).
print("Baseline:", round(1 - y_te.mean(), 2))

# 7) Treinamos a árvore (max_depth=3 limita a complexidade) e medimos o acerto no teste.
modelo = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_tr, y_tr)
print("Árvore:", round(accuracy_score(y_te, modelo.predict(X_te)), 2))

**Como ler:** o número do baseline é a nota que um modelo "preguiçoso" tira. A árvore só é boa se **supera** esse número, e a distância entre os dois é o que ela realmente aprendeu.

Sempre compare com um **baseline**. Um modelo com 80% de acerto parece bom, mas se o chute óbvio já dá 62%, o ganho real é bem menor.

**Para pensar:** as informações que usamos (classe, sexo, idade, tarifa) existiam *antes* do naufrágio. Isso é importante: um modelo útil só pode usar o que se sabe **no momento da decisão**.

## Parte 9: comunicando
Escreva **3 frases** para alguém sem conhecimento técnico (um familiar, por exemplo):

1. **O que encontramos?**
2. **Quão confiável é?** (nulos, tamanho da amostra, correlação ≠ causa)
3. **O que NÃO conseguimos afirmar?**

*Exemplo de estrutura:* "Nos dados, [grupo X] sobreviveu mais que [grupo Y]. Isso vale para [N] passageiros, mas [limitação]. Não dá para afirmar que [causa], porque [motivo]."

## Desafio final
Formule **sua própria pergunta** sobre este dataset e responda usando o ciclo completo.
Ideias: quem viajava sozinho (`alone`) sobreviveu mais? O porto de embarque (`embark_town`) importa? Famílias grandes (`sibsp`, `parch`) tiveram mais ou menos chance?

Checklist: ☐ conheci os dados ☐ vi os nulos ☐ procurei valores estranhos ☐ respondi com número ou gráfico ☐ separei correlação de causa ☐ escrevi as 3 frases